<a href="https://colab.research.google.com/github/waheba-husain/demandsense/blob/main/DemandSense.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [33]:
! pip install requests pandas

In [ ]:
from google.colab import userdata
userdata.get('API_KEY')

In [4]:
import requests
import pandas as pd

### **Loading Data**

In [5]:
BASE_URL = "https://api.eia.gov/v2/electricity/retail-sales/data/"

In [6]:
params = {
    "api_key": API_KEY,
    "frequency": "monthly",          # monthly data instead of annual
    "data[0]": "customers",          # include all desired metrics
    "data[1]": "price",
    "data[2]": "revenue",
    "data[3]": "sales",
    "facets[sectorid][]": "RES",     # filter to residential sector (RES = Residential)
    "start": "2010-01",              # can be None for all available data
    "end": None,                     # latest data
    "sort[0][column]": "period",
    "sort[0][direction]": "desc",
    "offset": 0,
    "length": 5000                   # pull up to 5000 rows
}

In [7]:
response = requests.get(BASE_URL, params=params)
data = response.json()

# Optional: Convert to pandas DataFrame
df = pd.DataFrame(data['response']['data'])
print(df.head())

    period stateid stateDescription sectorid   sectorName customers  price  \
0  2025-12      AK           Alaska      RES  residential    305423  25.54   
1  2025-12      AL          Alabama      RES  residential   2439723  16.01   
2  2025-12      AR         Arkansas      RES  residential   1500421  12.33   
3  2025-12      AZ          Arizona      RES  residential   3193720  15.46   
4  2025-12      CA       California      RES  residential  14730091  34.71   

      revenue       sales      customers-units              price-units  \
0    59.19987   231.77965  number of customers  cents per kilowatt-hour   
1   452.01356  2823.51089  number of customers  cents per kilowatt-hour   
2   195.86023  1588.25391  number of customers  cents per kilowatt-hour   
3   357.03192  2309.70764  number of customers  cents per kilowatt-hour   
4  2051.94171  5911.17633  number of customers  cents per kilowatt-hour   

     revenue-units             sales-units  
0  million dollars  million kilowat

In [8]:
print(df.columns.tolist())

['period', 'stateid', 'stateDescription', 'sectorid', 'sectorName', 'customers', 'price', 'revenue', 'sales', 'customers-units', 'price-units', 'revenue-units', 'sales-units']


# Storing data in a DB

## Schema design

In [9]:
import sqlite3

# Path to your SQLite database
DB_PATH = "energy.db"

# Connect to the database
conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

# Define your CREATE TABLE statement as a string
create_table_sql = """
CREATE TABLE IF NOT EXISTS electricity_consumption (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    date TEXT,
    hour INTEGER,
    state TEXT,
    state_name TEXT,
    sector_id TEXT,
    sector_name TEXT,
    customers INTEGER,
    price FLOAT,
    revenue FLOAT,
    sales FLOAT,
    customers_units TEXT,
    price_units TEXT,
    revenue_units TEXT,
    sales_units TEXT
)
"""

# Execute the SQL command
cursor.execute(create_table_sql)

# Save (commit) the changes and close the connection
conn.commit()
conn.close()

print("✅ Table created successfully.")



✅ Table created successfully.


## Preparing Data

In [10]:
def prepare_eia_df(raw_df):
    """Convert raw EIA response to cleaned format matching DB schema."""
    df = raw_df.copy()

    # Rename columns to match schema
    df.rename(columns={
        'period': 'date',
        'stateid': 'state',
        'stateDescription': 'state_name',
        'sectorid': 'sector_id',
        'sectorName': 'sector_name',
        'customers-units': 'customers_units',
        'price-units': 'price_units',
        'revenue-units': 'revenue_units',
        'sales-units': 'sales_units'
    }, inplace=True)

    df['hour'] = None  # Monthly data, keep hour null for future-proofing

    # Reorder columns to match schema
    columns = [
        'date', 'hour', 'state', 'state_name', 'sector_id', 'sector_name',
        'customers', 'price', 'revenue', 'sales',
        'customers_units', 'price_units', 'revenue_units', 'sales_units'
    ]
    return df[columns]


## Fetch & Save EIA Data

In [11]:
import sqlite3

conn = sqlite3.connect("energy.db")
df.to_sql("electricity_consumption", conn, if_exists="replace", index=False)
conn.close()
print("Table replaced with cleaned data.")

Table replaced with cleaned data.


In [13]:
df = prepare_eia_df(df)

In [14]:
from prophet import Prophet
import pandas as pd

# Ensure 'date' column exists and is datetime
df['date'] = pd.to_datetime(df['date'])

# Prepare dataframe for Prophet
forecast_df = df[['date','sales']].rename(columns={'date':'ds','sales':'y'})

# Initialize and train Prophet
model = Prophet(yearly_seasonality=True)
model.fit(forecast_df)

# Forecast next 12 months
future = model.make_future_dataframe(periods=12, freq='M')
forecast = model.predict(future)

# Show forecasted values
print(forecast[['ds','yhat','yhat_lower','yhat_upper']].tail())

INFO:prophet:Disabling weekly seasonality. Run prophet with weekly_seasonality=True to override this.
INFO:prophet:Disabling daily seasonality. Run prophet with daily_seasonality=True to override this.


           ds         yhat    yhat_lower    yhat_upper
88 2026-07-31  7725.636499 -11768.502375  29645.509379
89 2026-08-31  6423.733159 -14769.573026  26189.864775
90 2026-09-30  5253.574484 -14408.266423  26778.441156
91 2026-10-31  5220.117021 -16392.305299  24834.648063
92 2026-11-30  6283.717587 -15900.085309  26493.511796


/usr/local/lib/python3.12/dist-packages/prophet/forecaster.py:1875: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  dates = pd.date_range(


In [15]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense

In [16]:
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date')

sales_data = df[['sales']].values


In [17]:
scaler = MinMaxScaler()
scaled_sales = scaler.fit_transform(sales_data)

In [18]:
def create_sequences(data, window_size=12):
    X, y = [], []
    for i in range(window_size, len(data)):
        X.append(data[i-window_size:i, 0])
        y.append(data[i, 0])
    return np.array(X), np.array(y)

In [19]:
WINDOW_SIZE = 12
X, y = create_sequences(scaled_sales, WINDOW_SIZE)

X = X.reshape((X.shape[0], X.shape[1], 1))

In [20]:
split = int(0.8 * len(X))
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

In [21]:
model = Sequential([
    LSTM(64, activation='tanh', input_shape=(X.shape[1], 1)),
    Dense(1)
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [22]:
model.compile(optimizer='adam', loss='mse')


In [23]:
model.fit(
    X_train, y_train,
    epochs=20,
    batch_size=16,
    validation_data=(X_test, y_test),
    verbose=1
)

Epoch 1/20
250/250 ━━━━━━━━━━━━━━━━━━━━ 4s 8ms/step - loss: 0.0079 - val_loss: 0.0093
Epoch 2/20
250/250 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - loss: 0.0096 - val_loss: 0.0092
Epoch 3/20
250/250 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - loss: 0.0100 - val_loss: 0.0091
Epoch 4/20
250/250 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - loss: 0.0095 - val_loss: 0.0091
Epoch 5/20
250/250 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - loss: 0.0098 - val_loss: 0.0091
Epoch 6/20
250/250 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - loss: 0.0093 - val_loss: 0.0090
Epoch 7/20
250/250 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - loss: 0.0105 - val_loss: 0.0088
Epoch 8/20
250/250 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - loss: 0.0098 - val_loss: 0.0087
Epoch 9/20
250/250 ━━━━━━━━━━━━━━━━━━━━ 3s 10ms/step - loss: 0.0089 - val_loss: 0.0089
Epoch 10/20
250/250 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - loss: 0.0085 - val_loss: 0.0087
Epoch 11/20
250/250 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - loss: 0.0076 - val_loss: 0.0086
Epoch 12/20
250/250 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/ste

In [24]:
last_window = scaled_sales[-WINDOW_SIZE:].reshape(1, WINDOW_SIZE, 1)

In [25]:
future_predictions = []

for _ in range(12):
    pred = model.predict(last_window, verbose=0)[0][0]
    future_predictions.append(pred)
    last_window = np.append(last_window[:,1:,:], [[[pred]]], axis=1)

In [26]:
future_predictions = scaler.inverse_transform(
    np.array(future_predictions).reshape(-1,1)
)

In [27]:
future_dates = pd.date_range(
    start=df['date'].iloc[-1],
    periods=13,
    freq='ME'
)[1:]

lstm_forecast_df = pd.DataFrame({
    'date': future_dates,
    'lstm_sales_forecast': future_predictions.flatten()
})

In [29]:
import sqlite3

conn = sqlite3.connect("energy.db")
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS forecast (
    date TEXT,
    sales_forecast FLOAT,
    model TEXT
)
""")

conn.commit()
conn.close()
print("✅ Forecast table created")


✅ Forecast table created


In [28]:
import pandas as pd

conn = sqlite3.connect("energy.db")

# Prophet forecast
prophet_forecast_df = forecast[['ds','yhat']].rename(columns={'ds':'date','yhat':'sales_forecast'})
prophet_forecast_df['model'] = 'Prophet'
prophet_forecast_df.to_sql('forecast', conn, if_exists='append', index=False)

# LSTM forecast
lstm_forecast_df['model'] = 'LSTM'
lstm_forecast_df.rename(columns={'lstm_sales_forecast':'sales_forecast'}, inplace=True)
lstm_forecast_df.to_sql('forecast', conn, if_exists='append', index=False)

conn.commit()
conn.close()
print("✅ Forecasts saved to DB")


✅ Forecasts saved to DB


In [30]:
nl_to_sql = {
    "show next 6 months LSTM forecast": """
        SELECT date, sales_forecast
        FROM forecast
        WHERE model='LSTM' AND date > date('now')
        ORDER BY date ASC
        LIMIT 6
    """,
    "compare Prophet vs actual sales for 2025": """
        SELECT f.date, f.sales_forecast, d.sales AS actual_sales
        FROM forecast f
        JOIN electricity_consumption d ON f.date = d.date
        WHERE f.model='Prophet' AND strftime('%Y', f.date)='2025'
    """
}


In [31]:
def run_nl_query(nl_query):
    import sqlite3
    import pandas as pd

    if nl_query not in nl_to_sql:
        print("❌ Query not recognized. Try one from the predefined list.")
        return

    query = nl_to_sql[nl_query]
    conn = sqlite3.connect("energy.db")
    result = pd.read_sql(query, conn)
    conn.close()

    print(f"NL Query Result for: '{nl_query}'\n")
    print(result)


In [32]:
# Run any predefined natural language query
run_nl_query("show next 6 months LSTM forecast")
run_nl_query("show next 6 months Prophet forecast")


NL Query Result for: 'show next 6 months LSTM forecast'

                  date  sales_forecast
0  2026-03-31 00:00:00     -762.641907
1  2026-04-30 00:00:00     2349.738770
2  2026-05-31 00:00:00    39315.625000
3  2026-06-30 00:00:00    30258.578125
4  2026-07-31 00:00:00     2624.237061
5  2026-08-31 00:00:00     3484.725342
❌ Query not recognized. Try one from the predefined list.
